In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, year, month
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
import numpy as np
from pymongo import MongoClient

In [2]:
df = pd.read_csv('WeatherEvents_Jan2016-Dec2022.csv')

# data ingestion in mongodb


In [ ]:

print(f"Total rows to ingest: {len(df)}")


connection = MongoClient('mongodb://localhost:27017/')


db = connection['climate_agency_db']      
collection = db['weather_events_collection']

batch_size = 10000
total_rows = len(df)
data_dict = df.to_dict(orient='records')

try:
    collection.insert_many(data_dict)
    print("Data Successfully Ingested into MongoDB!")
except Exception as e:
    print(f"An error occurred during ingestion: {e}")




print(f"Total documents now stored in MongoDB: {collection.estimated_document_count()}")

Total rows to ingest: 8627181


In [ ]:
df.columns

In [ ]:
print(f"Total Rows and Columns: {df.shape}")

In [ ]:
print("\n--- Dataset Info ---")
df.info()

In [ ]:
print("\n--- Statistical Summary ---")
display(df.describe())

In [ ]:
df.isnull().sum()

In [ ]:
# Missing values ka sum nikalen
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

In [ ]:
# Ek dataframe mein combine karke dekhein
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Percentage (%)': missing_percentage})
display(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df, y='Type', order=df['Type'].value_counts().index, hue='Type', palette='viridis', legend=False)
plt.title('EDA: Frequency of Weather Event Types')
plt.xlabel('Total Count')
plt.ylabel('Event Type')
plt.show()

In [ ]:
# 2. Severity Distribution Countplot
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='Severity', order=df['Severity'].value_counts().index, hue='Severity', palette='Set2', legend=False)
plt.title('EDA: Distribution of Weather Severity Levels')
plt.xlabel('Severity')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
top_states = df['State'].value_counts().head(10)

sns.barplot(x=top_states.index, y=top_states.values, hue=top_states.index, palette='coolwarm', legend=False)
plt.title('EDA: Top 10 States with Highest Weather Events')
plt.xlabel('State')
plt.ylabel('Number of Events')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='LocationLng', y='LocationLat', hue='Severity', alpha=0.3, palette='Set1')
plt.title('Geographical Distribution of Weather Events by Severity')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Severity', y='Precipitation(in)', palette='Set3', hue='Severity', legend=False)
plt.title('Precipitation Levels across Weather Severity Categories')
plt.xlabel('Severity')
plt.ylabel('Precipitation (Inches)')
plt.show()

In [ ]:
# Ensure Year column exists
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'], errors='coerce')
df['Year'] = df['StartTime(UTC)'].dt.year

In [ ]:
    # Yearly Trend Line Plot
plt.figure(figsize=(12, 5))
yearly_trend = df['Year'].value_counts().sort_index()
sns.lineplot(x=yearly_trend.index, y=yearly_trend.values, marker='o', color='crimson', linewidth=2)
plt.title('Yearly Trend of Total Weather Events (2016-2022)')
plt.xlabel('Year')
plt.ylabel('Total Events')
plt.grid(True)
plt.show()

In [ ]:
    df_sample = df.sample(frac=0.1, random_state=42)
    
    plt.figure(figsize=(10, 6))
    
    # hue ke sath legend ko explicitly position kar dein taake warning na aaye
    sns.scatterplot(
        data=df_sample, 
        x='LocationLng', 
        y='LocationLat', 
        hue='Severity', 
        alpha=0.3, 
        palette='Set1'
    )
    
    plt.title('Geographical Distribution of Weather Events by Severity')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    
    # Legend ko right side par fix kar dein
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()